# Imports

In [1]:
import re
import optuna
import pickle

import numpy as np
import pandas as pd

from sklearn import set_config
from lightgbm import LGBMClassifier

from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import StackingClassifier, VotingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict

from feature_engine.selection import DropFeatures

/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
set_config(enable_metadata_routing=True)

# Loading Dataset

In [4]:
X_train = pd.read_parquet('../data/processed/X_train_logit.parquet')
y_train = pd.read_parquet('../data/processed/y_train.parquet')

X_test = pd.read_parquet('../data/processed/X_test_logit.parquet')

In [5]:
X_train.head()

,lgbm_logit,cat_logit,xgb_logit,hist_logit,lg_logit,knn_logit,ridge_logit,voting_lgbm_cat_xgb_hist_logit,voting_lg_ridge_logit,stacking_lgbm_with_knn_and_voting_lg_ridge_and_voting_lgbm_cat_xgb_hist_logit
0,1.402666,2.109990,2.149899,2.030346,1.440273,2.064999,0.500714,1.887848,0.925326,1.226844
1,0.616425,1.595299,1.584592,1.262993,0.799597,0.079907,0.335105,1.222563,0.561368,0.371762
2,0.023615,1.748857,1.106745,1.146720,0.235157,-1.325798,0.136991,0.929687,0.186260,0.066618
3,-6.367775,-5.125195,-4.946636,-4.912857,-6.083520,-34.538776,-1.247050,-5.198584,-2.069743,-6.799934
4,-0.438053,0.263538,-0.043875,-0.068158,-0.346862,-0.861416,-0.094611,-0.070366,-0.220629,-0.878595


In [6]:
X_test.head()

,lgbm_logit,cat_logit,xgb_logit,hist_logit,lg_logit,knn_logit,ridge_logit,voting_lgbm_cat_xgb_hist_logit,voting_lg_ridge_logit,stacking_lgbm_with_knn_and_voting_lg_ridge_and_voting_lgbm_cat_xgb_hist_logit
0,-5.379032,-3.409957,-3.759100,-3.888929,-5.042109,-34.538776,-1.272709,-3.899528,-34.538776,-5.055490
1,-5.795381,-4.426299,-4.187473,-3.950213,-2.684867,-2.019087,-0.626960,-4.396275,-34.538776,-5.602505
2,-5.574937,-4.441487,-4.232580,-4.454119,-5.395432,-34.538776,-1.335019,-4.564839,-34.538776,-5.656104
3,-1.635113,0.176618,0.347033,-0.058463,0.644945,-2.198160,0.190133,-0.222683,-0.304675,-1.163384
4,1.920299,3.398116,3.163860,3.161400,1.661579,34.539576,0.535153,2.745303,0.794179,2.146481


# Machine Learning

In [8]:
def objective(trial, X, y):

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    aucs = []

    params = {
        "objective": "binary",
        "n_estimators": trial.suggest_int("n_estimators", 30, 300),
        "learning_rate": trial.suggest_float("learning_rate", 0.003, 0.05, log=True),
        "max_depth": trial.suggest_int("max_depth", 1, 4),
        "num_leaves": trial.suggest_int("num_leaves", 2, 16),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 300),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-5, 100, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-5, 100, log=True),
        "verbosity": -1,
    }

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):

        X_train, X_valid = X.iloc[train_idx, :], X.iloc[valid_idx, :]
        y_train, y_valid = y.iloc[train_idx, 0], y.iloc[valid_idx, 0]

        model = LGBMClassifier(**params)
        model.fit(X_train, y_train)

        proba = model.predict_proba(X_valid)[:, 1]

        auc = roc_auc_score(y_valid, proba)
        aucs.append(auc)

        trial.report(np.mean(aucs), step=fold)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(aucs)


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42), pruner=optuna.pruners.MedianPruner(n_warmup_steps=2))
study.optimize(lambda trial: objective(trial, X_train, y_train), n_trials=1000, n_jobs=-1, show_progress_bar=True)


print("Best trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-05-19 13:54:51,966] A new study created in memory with name: no-name-b42b9c26-1eb4-48c4-8cdb-7b17f12642f7
  0%|                                                                                                                                                              | 0/1000 [00:18<?, ?it/s]

[I 2026-05-19 13:55:10,293] Trial 170 finished with value: 0.9498463750301551 and parameters: {'n_estimators': 288, 'learning_rate': 0.03893186343430399, 'max_depth': 4, 'num_leaves': 10, 'min_child_samples': 67, 'subsample': 0.7250795253582988, 'colsample_bytree': 0.6106818690522473, 'reg_alpha': 0.005432377538186629, 'reg_lambda': 0.008549304693634598}. Best is trial 155 with value: 0.9498604985212739.


Best trial: 0. Best value: 0.909042:   0%|                                                                                                               | 1/1000 [00:19<5:17:31, 19.07s/it]

[I 2026-05-19 13:55:11,010] Trial 0 finished with value: 0.9090419474642738 and parameters: {'n_estimators': 63, 'learning_rate': 0.003980527600692953, 'max_depth': 1, 'num_leaves': 2, 'min_child_samples': 163, 'subsample': 0.7925605842818102, 'colsample_bytree': 0.752455029983794, 'reg_alpha': 0.0440275925989403, 'reg_lambda': 0.4028640886469403}. Best is trial 0 with value: 0.9090419474642738.


Best trial: 0. Best value: 0.909042:   0%|                                                                                                               | 1/1000 [00:20<5:17:31, 19.07s/it]

...

[I 2026-05-19 15:12:21,032] Trial 993 finished with value: 0.949860172877894 and parameters: {'n_estimators': 266, 'learning_rate': 0.04745744459495787, 'max_depth': 4, 'num_leaves': 15, 'min_child_samples': 83, 'subsample': 0.9438322682182348, 'colsample_bytree': 0.6087132158974463, 'reg_alpha': 11.287507755552829, 'reg_lambda': 3.9339117390831917}. Best is trial 603 with value: 0.9498815886860695.


Best trial: 603. Best value: 0.949882: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████▉| 999/1000 [1:17:30<00:01,  1.56s/it]

[I 2026-05-19 15:12:22,774] Trial 996 finished with value: 0.9498672334572846 and parameters: {'n_estimators': 267, 'learning_rate': 0.04467063415385003, 'max_depth': 4, 'num_leaves': 15, 'min_child_samples': 98, 'subsample': 0.9607066727762179, 'colsample_bytree': 0.6057484805869686, 'reg_alpha': 9.899037788061916, 'reg_lambda': 3.6583320284042014}. Best is trial 603 with value: 0.9498815886860695.


Best trial: 603. Best value: 0.949882: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [1:17:31<00:00,  4.65s/it]

[I 2026-05-19 15:12:23,564] Trial 998 finished with value: 0.9498627418333221 and parameters: {'n_estimators': 268, 'learning_rate': 0.04456733190424611, 'max_depth': 4, 'num_leaves': 14, 'min_child_samples': 82, 'subsample': 0.9622271026771319, 'colsample_bytree': 0.6060328611590549, 'reg_alpha': 10.476951256022138, 'reg_lambda': 4.113478614845083}. Best is trial 603 with value: 0.9498815886860695.
Best trial score:
0.9498815886860695

Best params:
{'n_estimators': 272, 'learning_rate': 0.04331743813378749, 'max_depth': 4, 'num_leaves': 15, 'min_child_samples': 79, 'subsample': 0.849554199458748, 'colsample_bytree': 0.6452730806890068, 'reg_alpha': 3.4284318632046933, 'reg_lambda': 6.744375246788695}


In [9]:
stacking_lgbm = LGBMClassifier(**study.best_params).fit(X_train, y_train.PitNextLap)

[LightGBM] [Info] Number of positive: 87381, number of negative: 351759
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007971 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2550
[LightGBM] [Info] Number of data points in the train set: 439140, number of used features: 10
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.198982 -> initscore=-1.392668
[LightGBM] [Info] Start training from score -1.392668
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive g

# Submission

In [10]:
submission = pd.read_csv('../data/raw/sample_submission.csv')

In [11]:
submission['PitNextLap'] = stacking_lgbm.predict_proba(X_test)[:, 1]

In [12]:
submission.to_csv('../data/submission/submission.csv', index=False)

In [13]:
X_train.columns

Index(['lgbm_logit', 'cat_logit', 'xgb_logit', 'hist_logit', 'lg_logit',
       'knn_logit', 'ridge_logit', 'voting_lgbm_cat_xgb_hist_logit',
       'voting_lg_ridge_logit',
       'stacking_lgbm_with_knn_and_voting_lg_ridge_and_voting_lgbm_cat_xgb_hist_logit'],
      dtype='str')